# 4.1 Experimentos con bases de datos Weka (Ensemble & Distribucion OOD)
Prueba de inyección de sombras con K-Means y validación de extrapolación de un EnsembleRegressor.

In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.generators import ARFFDataGenerator
from src.shadows import ClosestClusterShadow
from src.splitter import GapSplitter
from src.orchestrator import RegressionOrchestator
from src.models import RandomForestModel, NeuralNetworkMCDropoutModel, BayesianNeuralNetworkModel
from src.ensemble import EnsembleRegressor
from src.noise import AbsoluteGaussianNoise


In [ ]:
datasets = ['bodyfat', 'cpu_act', 'housing', 'pbc', 'quake', 'cal_housing', 'house_16H', 'mv']
data_dir = '../data/weka_arff'

m_rf = RandomForestModel(n_estimators=50, max_depth=10)
m_drop = NeuralNetworkMCDropoutModel(hidden_dim=64, dropout_rate=0.2, epochs=300, n_iter=50)
m_bnn = BayesianNeuralNetworkModel(hidden_dim=32, epochs=300, n_iter=50)

ensemble = EnsembleRegressor(models=[m_rf, m_drop, m_bnn])

models = [m_rf, m_drop, m_bnn, ensemble]

shadow = ClosestClusterShadow(k_range=(5, 10), random_state=42)
splitter = GapSplitter(test_size=0.2, random_state=42)
dummy_noise = AbsoluteGaussianNoise(std_dev=0.0)

In [ ]:
results = []
dict_uncertainties = {}

for dataset_name in datasets:
    print(f"\n--- Datu-multzoa prozesatzen: {dataset_name} ---")
    generator = ARFFDataGenerator(dataset_name=dataset_name, data_dir=data_dir)
    
    dict_uncertainties[dataset_name] = {}
    
    for model in models:
        print(f"  Eredua exekutatzen: {model.get_name()}")
        orchestrator = RegressionOrchestator(generator=generator, splitter=splitter, model=model)
        
        try:
            artifacts = orchestrator.run(n_samples=-1, x_range=(0,0), noise_strategy=dummy_noise, shadow=shadow)
            metrics = artifacts.metrics
            
            unc_shadow = artifacts.uncertainties['shadow']
            unc_vis = artifacts.uncertainties['visible']
            dict_uncertainties[dataset_name][model.get_name()] = {'vis': unc_vis, 'shadow': unc_shadow}
            
            mean_unc_shadow = np.mean(unc_shadow) if artifacts.uncertainties is not None else 0.0
            
            results.append({
                'Dataset': dataset_name,
                'Model': model.get_name(),
                'MSE_Train': metrics.mse_train,
                'MSE_Vis': metrics.mse_test_visible,
                'MSE_Shadow': metrics.mse_test_shadow,
                'Uncertainty_Shadow': mean_unc_shadow
            })
        except Exception as e:
            print(f"  [ERROR] falló {model.get_name()} en {dataset_name}: {e}")

df_results = pd.DataFrame(results)
display(df_results)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

for dataset_name in datasets:
    fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
    fig.suptitle(f"KDE de Incertidumbres (Vis vs Shadow) - {dataset_name}", fontsize=14, y=1.05)
    
    for idx, model in enumerate(models):
        m_name = model.get_name()
        if m_name in dict_uncertainties[dataset_name]:
            unc_vis = dict_uncertainties[dataset_name][m_name]['vis']
            unc_shadow = dict_uncertainties[dataset_name][m_name]['shadow']
            
            sns.kdeplot(unc_vis, fill=True, color='blue', alpha=0.4, ax=axes[idx], label='Visible Data')
            sns.kdeplot(unc_shadow, fill=True, color='red', alpha=0.4, ax=axes[idx], label='Shadow Data (OOD)')
            axes[idx].set_title(m_name)
            axes[idx].set_xlabel("Incertidumbre Promedio")
            axes[idx].legend()
            
    plt.tight_layout()
    plt.show()